In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.animation import FFMpegWriter
from IPython.display import HTML, Video

In [2]:
# panning Doppler effect animation
def Doppler_pan(
    start=0, end=100, width=100, v_s=5, v=10, dt=0.5, 
    T=2, filled=False, faded=True, save=False
):
    '''
    `width` is the x range of the plot figure, measured from zero.
    The x limits of the plot will be 0 and `width`. `start` is
    the starting x position of the source, and end is the source's
    end position.
    '''
    
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=[10,3])
    ax.set_xlim(0, width)
    y_lim = 0.15 * width
    ax.set_ylim(-y_lim, y_lim)  
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    source, = ax.plot(start, 0, color='white', marker='o', ms=4)

    def update(frame):

        for patch in list(ax.patches):
            patch.remove()
            
        pos = positions[frame]

        source.set_data([pos], [0])

        wavefronts = []

        time_elapsed = frame * dt
        num_waves = int(time_elapsed / T) + 1

        if not filled:
            fill_waves = 0
        else:
            zone = max(width, end - start)
            fill_waves = int(1.5 * zone / (v * T)) - num_waves

        for wave in range(num_waves + fill_waves):
            
            emit_time = T * (wave - fill_waves)
            x = start + v_s * emit_time
            r = v * (time_elapsed - emit_time)

            fade = np.exp(-2 * r / width) if faded else 1

            wavefront = Circle(
                xy=(x,0), radius=r, fill=False, 
                edgecolor='yellow', alpha=fade
            )

            ax.add_patch(wavefront)
            wavefronts.append(wavefront)

        return [source, *wavefronts]

    frames = int((end - start) / (dt * v_s)) + 2
    positions = [start + v_s * frame * dt for frame in range(frames)]
        
    anim = FuncAnimation(fig, update, frames=frames, blit=True)

    plt.close(fig)

    fps = max(2, frames // 10)
    
    if not save:
        return HTML(anim.to_jshtml())
    
    if fps <= 20:
        anim.save('Doppler_pan.gif', writer=PillowWriter(fps=fps))
        return HTML(anim.to_jshtml())
    
    anim.save('Doppler_pan.mp4', writer=FFMpegWriter(fps=fps, codec='libx264'))
    return Video('Dopple_pan.mp4', embed=True)

In [3]:
# oscillating Doppler effect animation
def Doppler_osc(
    x_m=20, width=100, v=5, omega=np.pi, dt=0.1, 
    T=0.25, filled=False, faded=True, save=False
):
    
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=[10,3])
    plt.axis('off')
    ax.set_xlim(0, width)
    y_lim = 0.15 * width
    ax.set_ylim(-y_lim, y_lim)  
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

    centre = width / 2

    source, = ax.plot(centre, 0, color='white', marker='o', ms=4)
    
    frames = round(2 * np.pi / (omega * dt))
    if not filled: 
        frames *= 4
        
    positions = [centre + x_m * np.sin(omega * frame * dt) for frame in range(frames)]

    def update(frame):

        for patch in list(ax.patches):
            patch.remove()
            
        pos = positions[frame]

        source.set_data([pos], [0])

        wavefronts = []

        time_elapsed = frame * dt
        num_waves = int(time_elapsed / T) + 1

        if not filled:
            fill_waves = 0
        else:
            zone = max(width, width / 2 + x_m)
            fill_waves = int(1.5 * zone / (v * T)) - num_waves

        for wave in range(num_waves + fill_waves):
            
            emit_time = T * (wave - fill_waves)
            x = x_m * np.sin(omega * emit_time) + centre
            r = v * (time_elapsed - emit_time)

            fade = np.exp(-2 * r / width) if faded else 1

            wavefront = Circle(
                xy=(x,0), radius=r, fill=False, 
                edgecolor='yellow', alpha=fade
            )

            ax.add_patch(wavefront)
            wavefronts.append(wavefront)

        return [source, *wavefronts]
        
    anim = FuncAnimation(fig, update, frames=frames, blit=True)

    plt.close(fig)

    fps = max(2, frames // 5)
    
    if not save:
        return HTML(anim.to_jshtml())
    
    if fps <= 20:
        anim.save('Doppler_osc.gif', writer=PillowWriter(fps=fps))
        return HTML(anim.to_jshtml())
    
    anim.save('Doppler_osc.mp4', writer=FFMpegWriter(fps=fps, codec='libx264'))
    return Video('Doppler_osc.mp4', embed=True)

In [4]:
Doppler_osc(x_m=5, v=2.5*np.pi, omega=np.pi/2, dt=0.1, filled=True, save=False)